In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.sparse import csc_matrix
import folium
import warnings

# Suppress spatial/geometry warnings
warnings.filterwarnings('ignore')

target_cities = [
    "Kisumu, Kenya", 
    "Mombasa, Kenya", 
    "Nakuru, Kenya", 
    "Nairobi, Kenya"
]

# Broadened OSM tags to capture all structures and POIs
broad_tags = {
    'building': True,
    'landuse': 'residential',
    'amenity': True,
    'shop': True 
}

search_radius = 7500 # 7.5km radius captures peri-urban/informal outskirts
walk_limit = 2000    # 2km walking threshold
budget_per_city = 20 # Number of agents to place

city_data = {}



In [2]:

# PHASE 1: DATA EXTRACTION & REPROJECTION

print("Phase 1: Downloading Broadened Spatial Data...")
for city in target_cities:
    print(f"\n--> Extracting {city}...")
    try:
        # 1. Download pedestrian network via radius rather than admin boundary
        G = ox.graph_from_address(city, dist=search_radius, network_type='walk')
        
        # 2. Download all features within radius (handles OSMnx v1 and v2 API calls)
        if hasattr(ox, 'features') and hasattr(ox.features, 'features_from_address'):
            features = ox.features.features_from_address(city, tags=broad_tags, dist=search_radius)
        else:
            features = ox.features_from_address(city, tags=broad_tags, dist=search_radius)
        
        # 3. Native GeoPandas reprojection to local UTM to compute centroids accurately
        features_proj = features.to_crs(features.estimate_utm_crs())
        features_proj['geometry'] = features_proj.geometry.centroid
        features_points = features_proj.to_crs(features.crs) 
        
        # 4. Snap centroids to nearest network intersection node
        X = features_points.geometry.x.values
        Y = features_points.geometry.y.values
        
        if hasattr(ox, 'distance') and hasattr(ox.distance, 'nearest_nodes'):
            nearest_nodes = ox.distance.nearest_nodes(G, X, Y)
        else:
            nearest_nodes = ox.nearest_nodes(G, X, Y)
        
        features_points['nearest_node'] = nearest_nodes
        
        city_data[city] = {
            'graph': G,
            'demand': features_points,
            'nodes': list(G.nodes())
        }
        print(f"    Success: {len(G.nodes)} street nodes | {len(features_points)} demand structures.")
    except Exception as e:
        print(f"    Error processing {city}: {e}")





Phase 1: Downloading Broadened Spatial Data...

--> Extracting Kisumu, Kenya...
    Success: 6769 street nodes | 100169 demand structures.

--> Extracting Mombasa, Kenya...
    Success: 8444 street nodes | 53839 demand structures.

--> Extracting Nakuru, Kenya...
    Success: 8084 street nodes | 101021 demand structures.

--> Extracting Nairobi, Kenya...
    Success: 35472 street nodes | 212171 demand structures.


In [3]:
# PHASE 2: SPARSE DISTANCE MATRIX GENERATION

print("\nPhase 2: Generating Sparse Distance Matrices...")
for city, data in city_data.items():
    print(f"--> Computing ego-graphs for {city}...")
    G = data['graph']
    demand_df = data['demand']
    
    valid_nodes = list(demand_df['nearest_node'].unique())
    
    data['agent_node_list'] = valid_nodes
    data['demand_node_list'] = valid_nodes
    
    node_to_idx = {node: idx for idx, node in enumerate(valid_nodes)}
    
    row_indices = []
    col_indices = []
    distances = []
    
    for j_idx, agent_node in enumerate(valid_nodes):
        paths = nx.single_source_dijkstra_path_length(G, agent_node, cutoff=walk_limit, weight='length')
        for dest_node, dist in paths.items():
            if dest_node in node_to_idx:
                row_indices.append(node_to_idx[dest_node])
                col_indices.append(j_idx)
                distances.append(dist)
            
    cost_matrix = csc_matrix((distances, (row_indices, col_indices)), 
                             shape=(len(valid_nodes), len(valid_nodes)))
    data['cost_matrix'] = cost_matrix
    print(f"    Matrix built for {city} with shape {cost_matrix.shape}")



Phase 2: Generating Sparse Distance Matrices...
--> Computing ego-graphs for Kisumu, Kenya...
    Matrix built for Kisumu, Kenya with shape (6102, 6102)
--> Computing ego-graphs for Mombasa, Kenya...
    Matrix built for Mombasa, Kenya with shape (3379, 3379)
--> Computing ego-graphs for Nakuru, Kenya...
    Matrix built for Nakuru, Kenya with shape (7375, 7375)
--> Computing ego-graphs for Nairobi, Kenya...
    Matrix built for Nairobi, Kenya with shape (25229, 25229)


In [4]:


# PHASE 3: MCLP OPTIMIZATION & MST ROUTING

def calculate_agent_mst(G, chosen_agents):
    """Computes Minimum Spanning Tree linking selected agents for cash logistics."""
    agent_graph = nx.Graph()
    agent_list = list(chosen_agents)
    
    for agent in agent_list:
        agent_graph.add_node(agent)
        
    for i in range(len(agent_list)):
        for j in range(i + 1, len(agent_list)):
            node_a, node_b = agent_list[i], agent_list[j]
            try:
                dist = nx.shortest_path_length(G, source=node_a, target=node_b, weight='length')
                agent_graph.add_edge(node_a, node_b, weight=dist)
            except nx.NetworkXNoPath:
                pass
                
    return nx.minimum_spanning_tree(agent_graph, weight='weight')


optimization_results = {}

print("\nPhase 3: MCLP Optimization & MST Routing...")
for city, data in city_data.items():
    print(f"\n========== OPTIMIZING: {city.upper()} ==========")
    cost_matrix = data['cost_matrix']
    demand_node_list = data['demand_node_list']
    agent_node_list = data['agent_node_list']
    
    node_pop_series = data['demand'].groupby('nearest_node').size()
    pop_array = np.array([node_pop_series.get(node, 0) for node in demand_node_list])
    
    total_population = pop_array.sum()
    covered_mask = np.zeros(len(demand_node_list), dtype=bool)
    available_candidates = set(range(len(agent_node_list)))
    
    selected_agents = []
    population_covered = 0
    
    print(f"Total Demand Structures in {city}: {total_population}")
    
    for i in range(budget_per_city):
        best_candidate_idx = None
        max_pop_gained = 0
        best_new_covered_rows = []
        
        for candidate_idx in available_candidates:
            col = cost_matrix[:, candidate_idx]
            reachable_rows = col.indices
            distances_data = col.data
            
            within_radius = reachable_rows[distances_data <= walk_limit]
            new_uncovered = within_radius[~covered_mask[within_radius]]
            
            pop_gained = pop_array[new_uncovered].sum()
            
            if pop_gained > max_pop_gained:
                max_pop_gained = pop_gained
                best_candidate_idx = candidate_idx
                best_new_covered_rows = new_uncovered
                
        if max_pop_gained == 0 or best_candidate_idx is None:
            print(f"Stopping early. Maximum reach achieved at agent {i}.")
            break
            
        selected_agent_node = agent_node_list[best_candidate_idx]
        selected_agents.append(selected_agent_node)
        available_candidates.remove(best_candidate_idx)
        
        covered_mask[best_new_covered_rows] = True
        population_covered += max_pop_gained
        print(f"Agent {i+1} placed. Covers {max_pop_gained} unique structures.")
        
    mst = calculate_agent_mst(data['graph'], selected_agents)
    
    optimization_results[city] = {
        'chosen_agents': selected_agents,
        'mst': mst
    }
    
    coverage_pct = (population_covered / total_population * 100) if total_population > 0 else 0.0
    print(f"Optimization Complete! Coverage: {coverage_pct:.1f}%")




Phase 3: MCLP Optimization & MST Routing...

========== OPTIMIZING: KISUMU, KENYA ==========
Total Demand Structures in Kisumu, Kenya: 100169
Agent 1 placed. Covers 18016 unique structures.
Agent 2 placed. Covers 10874 unique structures.
Agent 3 placed. Covers 9522 unique structures.
Agent 4 placed. Covers 6083 unique structures.
Agent 5 placed. Covers 5463 unique structures.
Agent 6 placed. Covers 5033 unique structures.
Agent 7 placed. Covers 4513 unique structures.
Agent 8 placed. Covers 4473 unique structures.
Agent 9 placed. Covers 4105 unique structures.
Agent 10 placed. Covers 3933 unique structures.
Agent 11 placed. Covers 3079 unique structures.
Agent 12 placed. Covers 2827 unique structures.
Agent 13 placed. Covers 2385 unique structures.
Agent 14 placed. Covers 2218 unique structures.
Agent 15 placed. Covers 1889 unique structures.
Agent 16 placed. Covers 1474 unique structures.
Agent 17 placed. Covers 1323 unique structures.
Agent 18 placed. Covers 1131 unique structures.


In [6]:


# PHASE 4: VISUALIZATION DASHBOARD (STREET-LEVEL MST ROUTING)

print("\nPhase 4: Multi-Layer Visualization Dashboard (Street-Level Routing)...")

for city, data in city_data.items():
    if city not in optimization_results:
        continue
        
    G = data['graph']
    opt = optimization_results[city]
    chosen_agents = opt['chosen_agents']
    mst = opt['mst']
    
    if not chosen_agents:
        continue
        
    city_lat = G.nodes[chosen_agents[0]]['y']
    city_lon = G.nodes[chosen_agents[0]]['x']
    
    m = folium.Map(location=[city_lat, city_lon], zoom_start=12, tiles="cartodbpositron")
    
    fg_agents = folium.FeatureGroup(name="🏢 Agents & Coverage Radii", show=True)
    fg_routes = folium.FeatureGroup(name="🚚 Street-Level Cash Routes", show=True)

    for agent_node in chosen_agents:
        lat, lon = G.nodes[agent_node]['y'], G.nodes[agent_node]['x']
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            color='#1E8449',
            fill=True,
            fill_color='#2ECC71',
            fill_opacity=0.9,
            popup=f"Agent Node: {agent_node}"
        ).add_to(fg_agents)
        
        folium.Circle(
            location=[lat, lon],
            radius=walk_limit, 
            color='#2ECC71',
            fill=True,
            fill_color='#2ECC71',
            fill_opacity=0.08
        ).add_to(fg_agents)

    for u, v in mst.edges():
        try:
            street_path = nx.shortest_path(G, source=u, target=v, weight='length')
            route_coords = [[G.nodes[node]['y'], G.nodes[node]['x']] for node in street_path]
            
            path_length = sum(
                G.edges[street_path[i], street_path[i+1], 0]['length'] 
                for i in range(len(street_path) - 1)
            )
            
            folium.PolyLine(
                locations=route_coords,
                color='#D35400',
                weight=4,
                opacity=0.85,
                popup=f"Logistics Corridor: {path_length/1000:.2f} km"
            ).add_to(fg_routes)
            
        except (nx.NetworkXNoPath, KeyError):
            lat_u, lon_u = G.nodes[u]['y'], G.nodes[u]['x']
            lat_v, lon_v = G.nodes[v]['y'], G.nodes[v]['x']
            folium.PolyLine(
                locations=[[lat_u, lon_u], [lat_v, lon_v]],
                color='#E74C3C',
                weight=2,
                opacity=0.5,
                dash_array='4, 4'
            ).add_to(fg_routes)

    fg_routes.add_to(m)
    fg_agents.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    
    filename = f"{city.split(',')[0].lower()}_street_network.html"
    m.save(filename)
    print(f"    Saved interactive map: {filename}")


Phase 4: Multi-Layer Visualization Dashboard (Street-Level Routing)...
    Saved interactive map: kisumu_street_network.html
    Saved interactive map: mombasa_street_network.html
    Saved interactive map: nakuru_street_network.html
    Saved interactive map: nairobi_street_network.html


In [7]:

# PHASE 5: CSV METRICS EXPORT
print("\nPhase 5: Exporting Final Agent Locations to CSV...")
export_rows = []

for city, data in city_data.items():
    if city not in optimization_results:
        continue

    G = data['graph']
    cost_matrix = data['cost_matrix']
    demand_node_list = data['demand_node_list']
    agent_node_list = data['agent_node_list']
    chosen_agents = optimization_results[city]['chosen_agents']
    
    node_pop_series = data['demand'].groupby('nearest_node').size()
    pop_array = np.array([node_pop_series.get(node, 0) for node in demand_node_list])
    total_city_pop = pop_array.sum()
    
    node_to_idx = {node: idx for idx, node in enumerate(agent_node_list)}
    covered_mask = np.zeros(len(demand_node_list), dtype=bool)
    cumulative_covered = 0
    
    for rank, agent_node in enumerate(chosen_agents, 1):
        lat = G.nodes[agent_node]['y']
        lon = G.nodes[agent_node]['x']
        
        agent_idx = node_to_idx[agent_node]
        col = cost_matrix[:, agent_idx]
        reachable_rows = col.indices
        distances_data = col.data
        within_radius = reachable_rows[distances_data <= walk_limit]
        
        new_uncovered = within_radius[~covered_mask[within_radius]]
        marginal_gain = pop_array[new_uncovered].sum()
        
        covered_mask[new_uncovered] = True
        cumulative_covered += marginal_gain
        coverage_pct = (cumulative_covered / total_city_pop * 100) if total_city_pop > 0 else 0.0
        
        export_rows.append({
            'City': city.split(',')[0],
            'Agent_Rank': rank,
            'Node_ID': agent_node,
            'Latitude': round(lat, 6),
            'Longitude': round(lon, 6),
            'Marginal_Structures_Covered': int(marginal_gain),
            'Cumulative_Structures_Covered': int(cumulative_covered),
            'City_Total_Structures': int(total_city_pop),
            'Cumulative_Coverage_Pct': round(coverage_pct, 2)
        })

df_agents = pd.DataFrame(export_rows)
csv_filename = 'optimized_agent_locations.csv'
df_agents.to_csv(csv_filename, index=False)

print(f"Export Complete! Saved {len(df_agents)} entries to '{csv_filename}'.")


Phase 5: Exporting Final Agent Locations to CSV...
Export Complete! Saved 80 entries to 'optimized_agent_locations.csv'.
